In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: /Users/zaidashraf/Quantum-Secured-Threat-Intelligence-Pipeline


In [2]:
from pathlib import Path

BIO_DIR = (
    PROJECT_ROOT
    / "data"
    / "external"
    / "annoctr"
    / "AnnoCTR"
    / "ner_bio"
)

TRAIN_PATH = BIO_DIR / "train.bio"
TRAIN_EXT_PATH = BIO_DIR / "train_ext.bio"
TEST_PATH = BIO_DIR / "test.bio"

print("Train exists:", TRAIN_PATH.exists())
print("Extended train exists:", TRAIN_EXT_PATH.exists())
print("Test exists:", TEST_PATH.exists())

Train exists: True
Extended train exists: True
Test exists: True


In [3]:
def load_bio_file(path: str | Path) -> list[dict]:
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"BIO file not found: {path}")

    sentences = []
    tokens = []
    tags = []

    with path.open("r", encoding="utf-8") as file:
        for raw_line in file:
            line = raw_line.strip()

            if not line:
                if tokens:
                    sentences.append({
                        "tokens": tokens,
                        "ner_tags": tags,
                    })
                    tokens = []
                    tags = []
                continue

            parts = line.rsplit(maxsplit=1)

            if len(parts) != 2:
                continue

            token, tag = parts
            tokens.append(token)
            tags.append(tag)

    if tokens:
        sentences.append({
            "tokens": tokens,
            "ner_tags": tags,
        })

    return sentences

In [4]:
train_examples = load_bio_file(TRAIN_PATH)
train_ext_examples = load_bio_file(TRAIN_EXT_PATH)
test_examples = load_bio_file(TEST_PATH)

print("Train sentences:", len(train_examples))
print("Extended train sentences:", len(train_ext_examples))
print("Test sentences:", len(test_examples))

Train sentences: 7640
Extended train sentences: 42267
Test sentences: 3093


In [5]:
train_examples[0]

{'tokens': ['<DOCSTART>\tO\tO\tO\tO\tO'], 'ner_tags': ['O']}

In [6]:
from pathlib import Path
from datasets import load_dataset

NER_JSON_DIR = (
    PROJECT_ROOT
    / "data"
    / "external"
    / "annoctr"
    / "AnnoCTR"
    / "ner_json"
)

TRAIN_PATH = NER_JSON_DIR / "train.json"
DEV_PATH = NER_JSON_DIR / "dev.json"
TEST_PATH = NER_JSON_DIR / "test.json"

for path in [TRAIN_PATH, DEV_PATH, TEST_PATH]:
    print(path.name, "exists:", path.exists())

/opt/homebrew/Cellar/jupyterlab/4.4.3/libexec/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


train.json exists: True
dev.json exists: True
test.json exists: True


In [7]:
annoctr = load_dataset(
    "json",
    data_files={
        "train": str(TRAIN_PATH),
        "validation": str(DEV_PATH),
        "test": str(TEST_PATH),
    },
)

annoctr

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'text', 'ne_tags', 'nc_tags', 'te_tags', 'ce_tags', 'ci_tags', 'all_tags'],
        num_rows: 7570
    })
    validation: Dataset({
        features: ['id', 'tokens', 'text', 'ne_tags', 'nc_tags', 'te_tags', 'ce_tags', 'ci_tags', 'all_tags'],
        num_rows: 1550
    })
    test: Dataset({
        features: ['id', 'tokens', 'text', 'ne_tags', 'nc_tags', 'te_tags', 'ce_tags', 'ci_tags', 'all_tags'],
        num_rows: 3059
    })
})

In [8]:
for split_name, split_data in annoctr.items():
    print(f"\nSplit: {split_name}")
    print("Rows:", len(split_data))
    print("Columns:", split_data.column_names)


Split: train
Rows: 7570
Columns: ['id', 'tokens', 'text', 'ne_tags', 'nc_tags', 'te_tags', 'ce_tags', 'ci_tags', 'all_tags']

Split: validation
Rows: 1550
Columns: ['id', 'tokens', 'text', 'ne_tags', 'nc_tags', 'te_tags', 'ce_tags', 'ci_tags', 'all_tags']

Split: test
Rows: 3059
Columns: ['id', 'tokens', 'text', 'ne_tags', 'nc_tags', 'te_tags', 'ce_tags', 'ci_tags', 'all_tags']


In [9]:
example = annoctr["train"][0]

for key, value in example.items():
    print(f"\n{key}:")
    print(value)


id:
proofpoint_2021-05-18_threat-actors-exploit-microsoft-and__s0000

tokens:
['Security', 'Brief', ':', 'Threat', 'Actors', 'Exploit', 'Microsoft', 'and', 'Google', 'Platforms', 'to', 'Host', 'and', 'Send', 'Millions', 'of', 'Malicious', 'Messages']

text:
Security Brief: Threat Actors Exploit Microsoft and Google Platforms to Host and Send Millions of Malicious Messages

ne_tags:
['O', 'O', 'O', 'O', 'O', 'O', 'B-ORG', 'O', 'B-ORG', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']

nc_tags:
['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']

te_tags:
['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']

ce_tags:
['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']

ci_tags:
['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']

all_tags:
['O', 'O', 'O', 'O', 'O', 'O', 'B-ORG', 'O', 'B-ORG', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']


In [10]:
def validate_alignment(example):
    return len(example["tokens"]) == len(example["all_tags"])


for split_name in annoctr:
    invalid = [
        index
        for index, example in enumerate(annoctr[split_name])
        if not validate_alignment(example)
    ]

    print(
        split_name,
        "invalid examples:",
        len(invalid),
    )

train invalid examples: 0
validation invalid examples: 0
test invalid examples: 0


In [11]:
label_names = sorted({
    tag
    for split_name in annoctr
    for example in annoctr[split_name]
    for tag in example["all_tags"]
})

print("Number of labels:", len(label_names))
print(label_names)

Number of labels: 23
['B-CLICommand/CodeSnippet', 'B-CON', 'B-DATE', 'B-GROUP', 'B-LOC', 'B-MALWARE', 'B-ORG', 'B-SECTOR', 'B-TACTIC', 'B-TECHNIQUE', 'B-TOOL', 'I-CLICommand/CodeSnippet', 'I-CON', 'I-DATE', 'I-GROUP', 'I-LOC', 'I-MALWARE', 'I-ORG', 'I-SECTOR', 'I-TACTIC', 'I-TECHNIQUE', 'I-TOOL', 'O']


In [12]:
from collections import Counter

label_counts = Counter(
    tag
    for example in annoctr["train"]
    for tag in example["all_tags"]
)

label_counts

Counter({'O': 133845,
         'I-CLICommand/CodeSnippet': 3334,
         'I-TECHNIQUE': 2474,
         'I-DATE': 1176,
         'B-MALWARE': 1030,
         'B-ORG': 940,
         'B-CON': 903,
         'B-LOC': 733,
         'B-DATE': 709,
         'B-TECHNIQUE': 637,
         'I-TACTIC': 477,
         'B-TACTIC': 433,
         'I-ORG': 390,
         'B-GROUP': 387,
         'I-SECTOR': 315,
         'B-SECTOR': 287,
         'B-CLICommand/CodeSnippet': 213,
         'I-LOC': 181,
         'I-MALWARE': 144,
         'B-TOOL': 136,
         'I-GROUP': 92,
         'I-CON': 74,
         'I-TOOL': 30})

In [13]:
shown = 0

for example in annoctr["train"]:
    if any(tag != "O" for tag in example["all_tags"]):
        print(list(zip(
            example["tokens"],
            example["all_tags"],
        )))

        print("-" * 80)

        shown += 1

        if shown == 5:
            break

[('Security', 'O'), ('Brief', 'O'), (':', 'O'), ('Threat', 'O'), ('Actors', 'O'), ('Exploit', 'O'), ('Microsoft', 'B-ORG'), ('and', 'O'), ('Google', 'B-ORG'), ('Platforms', 'O'), ('to', 'O'), ('Host', 'O'), ('and', 'O'), ('Send', 'O'), ('Millions', 'O'), ('of', 'O'), ('Malicious', 'O'), ('Messages', 'O')]
--------------------------------------------------------------------------------
[('May', 'B-DATE'), ('18', 'I-DATE'), (',', 'I-DATE'), ('2021', 'I-DATE')]
--------------------------------------------------------------------------------
[('<', 'B-CLICommand/CodeSnippet'), ('!', 'I-CLICommand/CodeSnippet'), ('-', 'I-CLICommand/CodeSnippet'), ('-', 'I-CLICommand/CodeSnippet'), ('/', 'I-CLICommand/CodeSnippet'), ('*', 'I-CLICommand/CodeSnippet'), ('-', 'I-CLICommand/CodeSnippet'), ('-', 'I-CLICommand/CodeSnippet'), ('>', 'I-CLICommand/CodeSnippet'), ('<', 'I-CLICommand/CodeSnippet'), ('!', 'I-CLICommand/CodeSnippet'), ('[', 'I-CLICommand/CodeSnippet'), ('CDATA', 'I-CLICommand/CodeSnippet

In [14]:
label_names = sorted(label_names)

if "O" in label_names:
    label_names.remove("O")

label_names = ["O"] + label_names

label2id = {
    label: index
    for index, label in enumerate(label_names)
}

id2label = {
    index: label
    for label, index in label2id.items()
}

print(label2id)

{'O': 0, 'B-CLICommand/CodeSnippet': 1, 'B-CON': 2, 'B-DATE': 3, 'B-GROUP': 4, 'B-LOC': 5, 'B-MALWARE': 6, 'B-ORG': 7, 'B-SECTOR': 8, 'B-TACTIC': 9, 'B-TECHNIQUE': 10, 'B-TOOL': 11, 'I-CLICommand/CodeSnippet': 12, 'I-CON': 13, 'I-DATE': 14, 'I-GROUP': 15, 'I-LOC': 16, 'I-MALWARE': 17, 'I-ORG': 18, 'I-SECTOR': 19, 'I-TACTIC': 20, 'I-TECHNIQUE': 21, 'I-TOOL': 22}


In [15]:
def encode_labels(example):
    example["labels"] = [
        label2id[tag]
        for tag in example["all_tags"]
    ]

    return example


annoctr = annoctr.map(encode_labels)

annoctr["train"][0]

{'id': 'proofpoint_2021-05-18_threat-actors-exploit-microsoft-and__s0000',
 'tokens': ['Security',
  'Brief',
  ':',
  'Threat',
  'Actors',
  'Exploit',
  'Microsoft',
  'and',
  'Google',
  'Platforms',
  'to',
  'Host',
  'and',
  'Send',
  'Millions',
  'of',
  'Malicious',
  'Messages'],
 'text': 'Security Brief: Threat Actors Exploit Microsoft and Google Platforms to Host and Send Millions of Malicious Messages',
 'ne_tags': ['O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'B-ORG',
  'O',
  'B-ORG',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O'],
 'nc_tags': ['O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O'],
 'te_tags': ['O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O'],
 'ce_tags': ['O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O'],
 'ci_tags': ['O',

In [16]:
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
)

BASE_MODEL = "cisco-ai/SecureBERT2.0-base"

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    use_fast=True,
)

model = AutoModelForTokenClassification.from_pretrained(
    BASE_MODEL,
    num_labels=len(label_names),
    label2id=label2id,
    id2label=id2label,
)

print("Number of output labels:", model.config.num_labels)

Loading weights: 100%|█████████████████████| 136/136 [00:00<00:00, 5357.92it/s]
[transformers] ModernBertForTokenClassification LOAD REPORT from: cisco-ai/SecureBERT2.0-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Number of output labels: 23


In [17]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=512,
    )

    aligned_labels = []

    for batch_index, word_labels in enumerate(examples["labels"]):
        word_ids = tokenized_inputs.word_ids(
            batch_index=batch_index
        )

        previous_word_id = None
        label_ids = []

        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)

            elif word_id != previous_word_id:
                label_ids.append(word_labels[word_id])

            else:
                original_label_id = word_labels[word_id]
                original_label = id2label[original_label_id]

                if original_label.startswith("B-"):
                    inside_label = "I-" + original_label[2:]

                    label_ids.append(
                        label2id.get(
                            inside_label,
                            original_label_id,
                        )
                    )
                else:
                    label_ids.append(original_label_id)

            previous_word_id = word_id

        aligned_labels.append(label_ids)

    tokenized_inputs["labels"] = aligned_labels
    return tokenized_inputs

In [18]:
tokenized_annoctr = annoctr.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=annoctr["train"].column_names,
)

tokenized_annoctr

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 7570
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 1550
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 3059
    })
})

In [19]:
for split_name in tokenized_annoctr:
    invalid = 0

    for example in tokenized_annoctr[split_name]:
        if len(example["input_ids"]) != len(example["labels"]):
            invalid += 1

    print(split_name, "invalid aligned examples:", invalid)

train invalid aligned examples: 0
validation invalid aligned examples: 0
test invalid aligned examples: 0


In [20]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer
)

In [21]:
import evaluate
import numpy as np

seqeval = evaluate.load("seqeval")


def compute_metrics(eval_prediction):
    predictions, labels = eval_prediction

    predictions = np.argmax(predictions, axis=2)

    true_predictions = []
    true_labels = []

    for prediction_row, label_row in zip(predictions, labels):
        predicted_tags = []
        reference_tags = []

        for prediction_id, label_id in zip(
            prediction_row,
            label_row,
        ):
            if label_id == -100:
                continue

            predicted_tags.append(
                id2label[int(prediction_id)]
            )

            reference_tags.append(
                id2label[int(label_id)]
            )

        true_predictions.append(predicted_tags)
        true_labels.append(reference_tags)

    results = seqeval.compute(
        predictions=true_predictions,
        references=true_labels,
    )

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [22]:
from transformers import TrainingArguments

OUTPUT_DIR = PROJECT_ROOT / "models" / "ner_model"

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,
    report_to="none",
)

In [23]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_annoctr["train"],
    eval_dataset=tokenized_annoctr["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
training_result = trainer.train()
training_result

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': None, 'bos_token_id': None}.
/opt/homebrew/Cellar/jupyterlab/4.4.3/libexec/lib/python3.13/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


In [ ]:
validation_metrics = trainer.evaluate(
    tokenized_annoctr["validation"]
)

validation_metrics

In [ ]:
test_metrics = trainer.evaluate(
    tokenized_annoctr["test"]
)

test_metrics

In [8]:
import sys

!{sys.executable} -m pip install -U "accelerate>=1.1.0"

  Using cached accelerate-1.14.0-py3-none-any.whl.metadata (19 kB)
Using cached accelerate-1.14.0-py3-none-any.whl (389 kB)
